# HW04 作业 - 谢怡
**学号**：20234080418
**提交日期**：2026年6月


## 2 序列模型
### 2.1 理论计算题（一阶马尔可夫 + 拉普拉斯平滑）

**题目**：给定字符序列 `"ababc"`，词汇表为 `{'a','b','c'}`，采用一阶马尔可夫模型，使用加1平滑估计以下条件概率：

1. \( p(\text{a} \mid \text{b}) \)
2. \( p(\text{c} \mid \text{b}) \)

---

**解题步骤**：

**第一步：统计转移次数**

序列中相邻字符对（按顺序出现）为：
- a → b
- b → a
- a → b
- b → c

计数矩阵（行：前一个字符，列：后一个字符）：

| 前\后 | a | b | c |
| :---: | :---: | :---: | :---: |
| **a** | 0 | 2 | 0 |
| **b** | 1 | 0 | 1 |
| **c** | 0 | 0 | 0 |

**第二步：拉普拉斯平滑（加1）公式**
\[
p(x' \mid x) = \frac{\text{count}(x \to x') + 1}{\sum_{v \in \mathcal{V}} (\text{count}(x \to v) + 1)}
\]
分母中 \(|\mathcal{V}| = 3\)（词汇表大小）。

**第三步：计算具体值**

- 对于 **\( p(\text{a} \mid \text{b}) \)**：
  - 分子：\(\text{count(b→a)} + 1 = 1 + 1 = 2\)
  - 分母：\(\text{count(b→a)} + \text{count(b→b)} + \text{count(b→c)} + 3 = 1 + 0 + 1 + 3 = 5\)
  - **结果**：\(\boxed{\frac{2}{5} = 0.4}\)

- 对于 **\( p(\text{c} \mid \text{b}) \)**：
  - 分子：\(\text{count(b→c)} + 1 = 1 + 1 = 2\)
  - 分母同为 5
  - **结果**：\(\boxed{\frac{2}{5} = 0.4}\)

> 注：以 b 为条件的概率之和为 \(0.4 + 0.4 + 0.2 = 1.0\)，满足归一性。


In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    输入: text (str), n (滑动窗口长度)
    返回: vocab_dict, features_list, labels_list
    """
    # 1. 小写，去标点（保留字母和空格）
    text_clean = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    # 2. 分词
    words = text_clean.split()
    
    # 3. 构建词汇表（按频率排序，从0开始）
    freq = Counter(words)
    sorted_words = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features, labels = [], []
    for i in range(len(words) - n):
        features.append(words[i:i+n])
        labels.append(words[i+n])
    
    return vocab, features, labels

# ========== 测试 ==========
text = "The time machine"
n = 2
vocab, feats, labs = preprocess_text(text, n)
print("词汇表:", vocab)
print("特征:", feats)
print("标签:", labs)


词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


## 3 循环神经网络
### 3.1 理论计算题（BPTT 梯度推导）

**模型定义**：
\[
h_t = W_{hh}h_{t-1} + W_{hx}x_t,\quad o_t = W_{oh}h_t,\quad L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2
\]

**推导 \(\frac{\partial L}{\partial W_{hh}}\)**：

令损失对输出 \(o_t\) 的导数为 \(\delta_t^{(o)} = o_t - y_t\)。

根据链式法则，损失对隐藏状态 \(h_t\) 的梯度（从后向前传播）：
\[
\frac{\partial L}{\partial h_t} = W_{oh}^T (o_t - y_t) + W_{hh}^T \frac{\partial L}{\partial h_{t+1}} \quad (t < T)
\]
边界条件：
\[
\frac{\partial L}{\partial h_T} = W_{oh}^T (o_T - y_T)
\]

展开对 \(W_{hh}\) 的梯度，累加所有时间步的贡献：
\[
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^t \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial h_k} \cdot \frac{\partial h_k}{\partial W_{hh}}
\]
其中：
- \(\frac{\partial h_t}{\partial h_k} = \prod_{i=k+1}^{t} W_{hh} = W_{hh}^{t-k}\)
- \(\frac{\partial h_k}{\partial W_{hh}} = h_{k-1}^T\)

代入得最终表达式：
\[
\boxed{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^t \frac{\partial L}{\partial h_t} \left( W_{hh}^{t-k} \right)^T h_{k-1}^T}
\]

**梯度消失与爆炸条件**：
- 当权重矩阵 \(W_{hh}\) 的**谱半径**（最大特征值的绝对值）**大于 1** 时，\(W_{hh}^{t-k}\) 随 \(t-k\) 指数增长，导致**梯度爆炸**。
- 当谱半径**小于 1** 时，项指数衰减，导致**梯度消失**，使得长距离依赖难以学习。


In [2]:
import numpy as np

def rnn_cell_forward(x, h_prev, W_hx, W_hh, b):
    """
    x: (batch, input_size)
    h_prev: (batch, hidden_size)
    返回: h_next, cache
    """
    h_next = np.tanh(np.dot(h_prev, W_hh.T) + np.dot(x, W_hx.T) + b)
    cache = (x, h_prev, h_next, W_hx, W_hh, b)
    return h_next, cache

def rnn_cell_backward(dh_next, cache):
    """
    dh_next: 上游梯度 (batch, hidden)
    返回: dx, dh_prev, dW_hx, dW_hh, db
    """
    x, h_prev, h_next, W_hx, W_hh, b = cache
    batch_size, hidden_size = dh_next.shape
    input_size = x.shape[1]
    
    # tanh 导数: 1 - tanh^2
    dtanh = dh_next * (1 - h_next**2)
    
    # 计算各参数梯度
    dW_hh = np.dot(dtanh.T, h_prev)       # (hidden, hidden)
    dW_hx = np.dot(dtanh.T, x)            # (hidden, input)
    db = np.sum(dtanh, axis=0)            # (hidden,)
    dh_prev = np.dot(dtanh, W_hh)         # (batch, hidden)
    dx = np.dot(dtanh, W_hx)              # (batch, input)
    
    return dx, dh_prev, dW_hx, dW_hh, db

# ========== 测试 ==========
np.random.seed(42)
batch, in_size, hid_size = 2, 4, 3
x = np.random.randn(batch, in_size)
h_prev = np.random.randn(batch, hid_size)
W_hx = np.random.randn(hid_size, in_size)
W_hh = np.random.randn(hid_size, hid_size)
b = np.random.randn(hid_size)

h_next, cache = rnn_cell_forward(x, h_prev, W_hx, W_hh, b)
dh_next = np.random.randn(batch, hid_size)
dx, dh_prev, dW_hx, dW_hh, db = rnn_cell_backward(dh_next, cache)

print("前向隐藏状态形状:", h_next.shape)
print("dx 形状:", dx.shape)
print("dh_prev 形状:", dh_prev.shape)
print("dW_hx 形状:", dW_hx.shape)
print("dW_hh 形状:", dW_hh.shape)
print("db 形状:", db.shape)


前向隐藏状态形状: (2, 3)
dx 形状: (2, 4)
dh_prev 形状: (2, 3)
dW_hx 形状: (3, 4)
dW_hh 形状: (3, 3)
db 形状: (3,)


## 4 高级循环神经网络
### 4.1 理论计算题（深度双向 RNN 参数量）

**参数符号**：
- \(L\)：层数
- \(H\)：每层隐藏单元数
- \(D\)：输入维度
- \(O\)：输出维度（仅最后输出层）

**逐层参数量计算**：

**第 1 层（双向）**：
- 前向 RNN：输入权重 \(H \times D\)，隐藏权重 \(H \times H\)，偏置 \(H\)
- 反向 RNN：参数独立，同上
- 小计：\(2 \times (HD + H^2 + H)\)

**第 2 至 \(L\) 层（双向）**：
- 每层的输入维度为前一层的双向拼接：\(2H\)
- 每个方向（前/后）：输入权重 \(H \times 2H\)，隐藏权重 \(H \times H\)，偏置 \(H\)
- 每层双向小计：\(2 \times (2H^2 + H^2 + H) = 2 \times (3H^2 + H)\)
- \((L-1)\) 层小计：\((L-1) \times 2 \times (3H^2 + H)\)

**输出层**：
- 输入为最后一层的双向拼接：\(2H\)
- 权重 \(2H \times O\)，偏置 \(O\)
- 小计：\(2HO + O\)

**总参数量表达式**：
\[
\boxed{\text{Total} = 2(HD + H^2 + H) + (L-1) \cdot 2(3H^2 + H) + (2HO + O)}
\]


In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, 
                          bidirectional=True, batch_first=False)
        self.hidden_dim = hidden_dim
        
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        返回:
            outputs: (seq_len, batch, 2*hidden_dim)
            final_state: (batch, 2*hidden_dim)
        """
        outputs, h_n = self.rnn(X)  # h_n: (num_layers*2, batch, hidden_dim)
        # 取最后一层前向和反向的隐藏状态并拼接
        final_forward = h_n[-2, :, :]   # (batch, hidden_dim)
        final_backward = h_n[-1, :, :]  # (batch, hidden_dim)
        final_state = torch.cat([final_forward, final_backward], dim=1)  # (batch, 2*hidden_dim)
        return outputs, final_state

# ========== 测试 ==========
seq_len, batch, input_dim, hidden_dim = 5, 3, 4, 6
X = torch.randn(seq_len, batch, input_dim)
encoder = BiRNNEncoder(input_dim, hidden_dim, num_layers=1)
outputs, final_state = encoder(X)

print("输出 outputs 形状:", outputs.shape)
print("最终状态 final_state 形状:", final_state.shape)


输出 outputs 形状: torch.Size([5, 3, 12])
最终状态 final_state 形状: torch.Size([3, 12])


## 5 嵌入向量
### 5.1 理论计算题（Skip-gram 负采样损失函数）

**给定**：中心词 \(w_c\)，上下文词 \(w_o\)，从噪声分布 \(P_n(w)\) 采样 \(K\) 个负样本 \(w_{n_k}\)。

**负采样下的目标函数（最大化对数似然）**：
\[
\mathcal{L} = \log \sigma(\mathbf{v}_c^T \mathbf{u}_o) + \sum_{k=1}^{K} \mathbb{E}_{w_{n_k} \sim P_n(w)} \left[ \log \sigma(-\mathbf{v}_c^T \mathbf{u}_{n_k}) \right]
\]

**等价的最小化损失函数**：
\[
\boxed{\mathcal{L}_{\text{loss}} = -\log \sigma(\mathbf{v}_c^T \mathbf{u}_o) - \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_c^T \mathbf{u}_{n_k})}
\]

**负样本采样方式**：
- 从噪声分布 \(P_n(w)\) 中独立采样 \(K\) 个词。
- 常用噪声分布为**词频的 \(3/4\) 次方**（即 \(P_n(w) \propto \text{freq}(w)^{3/4}\)），这种分布能够平衡高频词和低频词的采样概率。
- 采样时需排除当前正样本上下文词 \(w_o\)。


In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_idx, W_in, W_out):
    """
    context_indices: (batch_size, context_size)  上下文词索引
    target_idx: (batch_size,)                    中心词索引
    W_in: (V, d)  输入权重矩阵
    W_out: (d, V) 输出权重矩阵
    返回: loss (标量)
    """
    # 1. 获取上下文嵌入并求平均
    emb = W_in[context_indices]          # (batch, context_size, d)
    h = emb.mean(dim=1)                  # (batch, d)
    
    # 2. 计算输出 logits
    logits = torch.matmul(h, W_out)      # (batch, V)
    
    # 3. 交叉熵损失
    loss = F.cross_entropy(logits, target_idx)
    return loss

# ========== 测试 ==========
torch.manual_seed(42)
V, d, batch_size, context_size = 10, 5, 4, 3

W_in = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)
context = torch.randint(0, V, (batch_size, context_size))
target = torch.randint(0, V, (batch_size,))

loss = cbow_forward(context, target, W_in, W_out)
print("CBOW 交叉熵损失值:", loss.item())


CBOW 交叉熵损失值: 2.686213493347168


## 6 注意力机制
### 6.1 理论计算题（缩放点积注意力计算过程）

**已知**：
- \(Q \in \mathbb{R}^{2 \times 4}\)
- \(K \in \mathbb{R}^{3 \times 4}\)
- \(V \in \mathbb{R}^{3 \times 5}\)
- \(d_k = 4\)

**计算三步骤**：

**步骤 1：计算得分矩阵（Scores）**
\[
\text{Scores} = \frac{Q K^T}{\sqrt{d_k}} = \frac{Q K^T}{2}
\]
形状为 \(2 \times 3\)。

**步骤 2：按行 Softmax（每行和为 1）**
\[
\text{Attention Weights} = \text{softmax}(\text{Scores}) \quad (\text{按行计算})
\]
形状为 \(2 \times 3\)。

**步骤 3：加权求和得到输出**
\[
\text{Output} = \text{Attention Weights} \times V
\]
形状为 \(2 \times 5\)。

**Python 数值演示（随机生成具体矩阵）**：


In [5]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
Q = torch.randn(2, 4)
K = torch.randn(3, 4)
V = torch.randn(3, 5)
d_k = 4

scores = torch.matmul(Q, K.T) / (d_k ** 0.5)
attn_weights = F.softmax(scores, dim=-1)
output = torch.matmul(attn_weights, V)

print("得分矩阵形状:", scores.shape)
print("注意力权重形状:", attn_weights.shape)
print("最终输出形状:", output.shape)
print("\n得分矩阵 Scores:\n", scores)
print("\n注意力权重 Attention Weights:\n", attn_weights)
print("\n最终输出 Output:\n", output)


得分矩阵形状: torch.Size([2, 3])
注意力权重形状: torch.Size([2, 3])
最终输出形状: torch.Size([2, 5])

得分矩阵 Scores:
 tensor([[ 0.2067,  0.3803,  0.0517],
        [ 0.6280, -0.4697, -0.7661]])

注意力权重 Attention Weights:
 tensor([[0.3283, 0.3905, 0.2812],
        [0.6322, 0.2109, 0.1568]])

最终输出 Output:
 tensor([[ 0.7610, -0.3948,  0.0536,  0.3988, -0.8415],
        [ 0.5821,  0.4428,  0.4574,  0.5706, -0.3782]])


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        返回: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影并拆分为多头
        Q = self.W_Q(X).view(seq_len, batch, self.num_heads, self.d_k)
        K = self.W_K(X).view(seq_len, batch, self.num_heads, self.d_k)
        V = self.W_V(X).view(seq_len, batch, self.num_heads, self.d_k)
        
        # 交换维度: (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 2. 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (batch, heads, seq_len, d_k)
        
        # 3. 拼接多头
        attn_output = attn_output.permute(2, 0, 1, 3).contiguous()  # (seq_len, batch, heads, d_k)
        concat = attn_output.view(seq_len, batch, self.d_model)
        
        # 4. 最终线性层
        output = self.W_O(concat)
        return output

# ========== 测试 ==========
d_model, num_heads = 4, 2
seq_len, batch = 6, 3
X = torch.randn(seq_len, batch, d_model)

mha = MultiHeadAttention(d_model, num_heads)
out = mha(X)

print("输入 X 形状:", X.shape)
print("输出 out 形状:", out.shape)


输入 X 形状: torch.Size([6, 3, 4])
输出 out 形状: torch.Size([6, 3, 4])
